# Milestone2 • ProRes GPU worker
Select a GPU runtime, Run all and authorize Drive. One worker per job. Interruptions may lose the current uncommitted chunk, not previously uploaded chunks. GPU allocation is not guaranteed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
JOB = '/content/drive/MyDrive/JILP_MILESTONES_2026/colab-jobs/floating-prores-v1'


In [ ]:
import pathlib, subprocess, urllib.request, hashlib, tarfile, json, shutil
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
base='https://download.blender.org/release/Blender5.0/'
name='blender-5.0.1-linux-x64.tar.xz'
archive=Path('/content')/name
exe=Path('/content/blender-5.0.1-linux-x64/blender')
if not exe.is_file():
    urllib.request.urlretrieve(base+name,archive)
    expected=next(line.split()[0] for line in urllib.request.urlopen(base+'blender-5.0.1.sha256').read().decode().splitlines() if line.endswith(name))
    with archive.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==expected, 'Blender checksum mismatch'
    subprocess.run(['tar','-xJf',str(archive),'-C','/content'],check=True)

import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'FFmpeg required'


In [ ]:
import tarfile, os
job=Path(JOB)
cfg=json.loads((job/'job.json').read_text())
assert cfg['version']==2
work=Path('/content/milestone2-worker')/cfg['sha256'][:16]
work.mkdir(parents=True,exist_ok=True)
bundle=work/'input.tar'
if not bundle.exists(): shutil.copy2(job/'input.tar',bundle)
with bundle.open('rb') as f: assert hashlib.file_digest(f,'sha256').hexdigest()==cfg['sha256']
with tarfile.open(bundle) as t: t.extractall(work,filter='data')
meta=json.loads((work/'cache/manifest.json').read_text())
assert meta['count']==cfg['count'] and meta['settings']['buildHash']==cfg['buildHash']
out=work/'render.prores'; out.mkdir(exist_ok=True)
remote=job/'render.prores'; remote.mkdir(exist_ok=True)
signature=json.dumps(cfg,sort_keys=True)
marker=remote/'job-settings.json'
if marker.exists(): assert marker.read_text()==signature
else: marker.write_text(signature)
# Restore completed chunks and corresponding depth files only.
for mark in sorted((remote/'chunks').glob('chunk-*.json')):
    d=json.loads(mark.read_text())
    for i in range(d['start'],d['start']+d['count']):
        rel=Path('depth')/f'frame-{i:06d}.png'
        (out/rel).parent.mkdir(exist_ok=True)
        shutil.copy2(remote/rel,out/rel)
    (out/'chunks').mkdir(exist_ok=True)
    shutil.copy2(mark.with_suffix('.mov'),out/'chunks'/mark.with_suffix('.mov').name)
    shutil.copy2(mark,out/'chunks'/mark.name)
print('Verified cache:',cfg['scene'],cfg['count'],'frames')


In [ ]:
# Render locally on Colab SSD; upload each completed chunk atomically.
import threading, time
def publish(src,dst):
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.with_name(dst.name+'.uploading')
    shutil.copy2(src,tmp)
    os.replace(tmp,dst)
def sync_chunks():
    for mark in sorted((out/'chunks').glob('chunk-*.json')):
        dst=remote/'chunks'/mark.name
        if dst.exists(): continue
        d=json.loads(mark.read_text())
        for i in range(d['start'],d['start']+d['count']):
            rel=Path('depth')/f'frame-{i:06d}.png'
            publish(out/rel,remote/rel)
        publish(mark.with_suffix('.mov'),dst.with_suffix('.mov'))
        publish(mark,dst)
        print('Uploaded chunk',d['start'],flush=True)
cmd=[str(exe),'-b','--factory-startup','--python-exit-code','1','-P',str(work/'blender/render_cache.py'),'--','--cache',str(work/'cache'),'--output',str(out),'--format','PNG','--passes','beauty','--prores','--device','CUDA','--ffmpeg',shutil.which('ffmpeg'),'--ffprobe',shutil.which('ffprobe')]
for k in ('width','height','samples','bounces','noise','min_samples','diffuse','glossy','chunk_size','depth_far'):
    cmd+=['--'+k.replace('_','-'),str(cfg[k])]
proc=subprocess.Popen(cmd)
try:
    while proc.poll() is None:
        sync_chunks()
        time.sleep(5)
finally:
    if proc.poll() is None:
        proc.terminate()
        proc.wait()
    sync_chunks()
assert proc.returncode==0, 'Render interrupted; rerun to resume'
for name in ('depth-metadata.json','media.json'):
    publish(out/name,remote/name)
print('Complete:',remote)
